In [1]:
import os

DATASET_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new"
for root, dirs, files in os.walk(DATASET_ROOT):
    depth = root.replace(DATASET_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in files:
        print(f"{indent}  {f}")

caai-revision-splits-new/
  vae_credit_card/
    config.json
    scores_id2.parquet
    scores_ood.parquet
    scores_id.parquet
    model.pt
  ae_credit_card/
    config.json
    scores_id2.parquet
    scores_ood.parquet
    scores_id.parquet
    model.pt


In [2]:
import os

for name in os.listdir("/kaggle/input/datasets"):
    print(name)

zlemglsmklkaya


In [3]:
import os

user_dir = "/kaggle/input/datasets/zlemglsmklkaya"
for name in os.listdir(user_dir):
    print(name)

caai-revision-splits-new


In [4]:
import os

user_dir = "/kaggle/input/datasets/zlemglsmklkaya"
for name in os.listdir(user_dir):
    print(name)

caai-revision-splits
caai-revision-splits-new


In [5]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SPLITS_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cic_scenarios = {}
for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(SPLITS_ROOT, "cic_ids2017", scen)
    cic_scenarios[scen] = {
        "train": pd.read_parquet(os.path.join(d, "train.parquet")),
        "id": pd.read_parquet(os.path.join(d, "id.parquet")),
        "id2": pd.read_parquet(os.path.join(d, "id2.parquet")),
        "ood": pd.read_parquet(os.path.join(d, "ood.parquet")),
    }

with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    cic_config = json.load(f)
CIC_FEATURES = cic_config["model_features"]

print(f"\nCIC-IDS2017 features ({len(CIC_FEATURES)}):", CIC_FEATURES[:5], "...")
for scen, d in cic_scenarios.items():
    print(f"{scen}: Train={d['train'].shape}, ID={d['id'].shape}, ID2={d['id2'].shape}, OOD={d['ood'].shape}")

Device: cuda
GPU: Tesla T4

CIC-IDS2017 features (67): ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets'] ...
scenario_T: Train=(910016, 82), ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
scenario_A: Train=(1190196, 82), ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)


In [6]:
def make_fit_val_split(train_df, val_frac=0.10, seed=SEED):
    shuffled = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_val = int(len(shuffled) * val_frac)
    val = shuffled.iloc[:n_val].copy()
    fit = shuffled.iloc[n_val:].copy()
    return fit, val

splits_for_training = {}
for scen, d in cic_scenarios.items():
    fit, val = make_fit_val_split(d["train"])
    splits_for_training[scen] = {"fit": fit, "val": val, "features": CIC_FEATURES}
    print(f"{scen}: fit={len(fit)}, val={len(val)}")

FITVAL_DIR = "/kaggle/working/fit_val_splits"
for name, d in splits_for_training.items():
    p = os.path.join(FITVAL_DIR, name)
    os.makedirs(p, exist_ok=True)
    d["fit"].to_parquet(os.path.join(p, "fit.parquet"), index=False)
    d["val"].to_parquet(os.path.join(p, "val.parquet"), index=False)
print(f"\nSaved fit/val splits to {FITVAL_DIR}")

scenario_T: fit=819015, val=91001
scenario_A: fit=1071177, val=119019

Saved fit/val splits to /kaggle/working/fit_val_splits


In [7]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def train_autoencoder(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = Autoencoder(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit),
        batch_size=train_config["batch_size"], shuffle=True
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon = model(batch)
            loss = loss_fn(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            val_recon = model(X_val)
            val_loss = loss_fn(val_recon, X_val).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - train_config["min_delta"]:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_loss, epoch


class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.encoder_body = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        dec_layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.decoder_body = nn.Sequential(*dec_layers)
        self.fc_out_mu = nn.Linear(prev, input_dim)
        self.fc_out_logvar = nn.Linear(prev, input_dim)

    def encode(self, x):
        h = self.encoder_body(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_body(z)
        out_mu = self.fc_out_mu(h)
        out_logvar = torch.clamp(self.fc_out_logvar(h), min=-10, max=10)
        return out_mu, out_logvar

    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        out_mu, out_logvar = self.decode(z)
        return out_mu, out_logvar, mu_z, logvar_z


def vae_elbo_loss(x, out_mu, out_logvar, mu_z, logvar_z):
    recon_nll = 0.5 * (out_logvar + ((x - out_mu) ** 2) / torch.exp(out_logvar) + np.log(2 * np.pi))
    recon_nll = recon_nll.sum(dim=1).mean()
    kl = -0.5 * torch.sum(1 + logvar_z - mu_z.pow(2) - logvar_z.exp(), dim=1).mean()
    return recon_nll + kl, recon_nll, kl


def train_vae(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = VAE(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit),
        batch_size=train_config["batch_size"], shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])

    best_val_elbo = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_elbo": [], "val_elbo": [], "val_recon_nll": [], "val_kl": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out_mu, out_logvar, mu_z, logvar_z = model(batch)
            loss, _, _ = vae_elbo_loss(batch, out_mu, out_logvar, mu_z, logvar_z)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_elbo = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            out_mu, out_logvar, mu_z, logvar_z = model(X_val)
            val_elbo, val_recon_nll, val_kl = vae_elbo_loss(X_val, out_mu, out_logvar, mu_z, logvar_z)
            val_elbo, val_recon_nll, val_kl = val_elbo.item(), val_recon_nll.item(), val_kl.item()

        history["train_elbo"].append(train_elbo)
        history["val_elbo"].append(val_elbo)
        history["val_recon_nll"].append(val_recon_nll)
        history["val_kl"].append(val_kl)

        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_elbo={train_elbo:.4f}  val_elbo={val_elbo:.4f}  "
                  f"(recon_nll={val_recon_nll:.4f}, kl={val_kl:.4f})")

        if val_elbo < best_val_elbo - train_config["min_delta"]:
            best_val_elbo = val_elbo
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_elbo, epoch

print("Classes and training functions defined.")

Classes and training functions defined.


In [8]:
import time

ARCH_CONFIG_CIC_PROBE = {
    "hidden_dims": [48, 32],
    "latent_dim": 16,
}

probe_train_config = {
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 256,
    "max_epochs": 2,
    "patience": 999,       # disabled — we just want raw timing, not convergence
    "min_delta": 1e-5,
    "early_stopping_criterion": "n/a - timing probe only",
    "seed": SEED,
    "loss": "MSE",
}

t0 = time.time()
_ = train_autoencoder(
    fit_df=splits_for_training["scenario_T"]["fit"],
    val_df=splits_for_training["scenario_T"]["val"],
    feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC_PROBE,
    train_config=probe_train_config,
    device=DEVICE,
    log_every=1,
)
elapsed = time.time() - t0
print(f"\n2 epochs took {elapsed:.1f}s -> ~{elapsed/2:.1f}s/epoch")
print(f"Rough estimate for 100 epochs: {elapsed/2*100/60:.1f} min")
print(f"Rough estimate for 200 epochs: {elapsed/2*200/60:.1f} min")

  Epoch   1  train_loss=0.256115  val_loss=0.138873
  Epoch   2  train_loss=0.143355  val_loss=0.045094

2 epochs took 25.2s -> ~12.6s/epoch
Rough estimate for 100 epochs: 21.0 min
Rough estimate for 200 epochs: 41.9 min


In [9]:
ARCH_CONFIG_CIC = {
    "hidden_dims": [48, 32],
    "latent_dim": 16,
    "architecture_rationale": "Funnel (67->48->32->16->32->48->67); width scaled relative to 67 input features, not tuned",
}

TRAIN_CONFIG_CIC = {
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 256,
    "max_epochs": 200,
    "patience": 10,
    "min_delta": 1e-5,
    "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
    "seed": SEED,
    "loss": "MSE",
}
print(json.dumps(ARCH_CONFIG_CIC, indent=2))
print(json.dumps(TRAIN_CONFIG_CIC, indent=2))

print("\nTraining Autoencoder — CIC-IDS2017 scenario_T")
ae_cic_T, history_ae_cic_T, best_val_loss_ae_cic_T, stopped_epoch_ae_cic_T = train_autoencoder(
    fit_df=splits_for_training["scenario_T"]["fit"],
    val_df=splits_for_training["scenario_T"]["val"],
    feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_CIC,
    train_config=TRAIN_CONFIG_CIC,
    device=DEVICE,
    log_every=10,
)
print(f"\nFinished at epoch {stopped_epoch_ae_cic_T}, best val_loss={best_val_loss_ae_cic_T:.6f}")

{
  "hidden_dims": [
    48,
    32
  ],
  "latent_dim": 16,
  "architecture_rationale": "Funnel (67->48->32->16->32->48->67); width scaled relative to 67 input features, not tuned"
}
{
  "optimizer": "Adam",
  "learning_rate": 0.001,
  "batch_size": 256,
  "max_epochs": 200,
  "patience": 10,
  "min_delta": 1e-05,
  "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
  "seed": 42,
  "loss": "MSE"
}

Training Autoencoder — CIC-IDS2017 scenario_T
  Epoch   1  train_loss=0.246407  val_loss=0.071194
  Epoch  10  train_loss=0.055469  val_loss=0.023531
  Epoch  20  train_loss=0.041004  val_loss=0.013451
  Epoch  30  train_loss=0.025686  val_loss=0.010673
  Early stopping at epoch 34 (no improvement for 10 epochs)

Finished at epoch 34, best val_loss=0.007598


In [10]:
def compute_reconstruction_error(model, df, feature_cols, device):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        recon = model(X)
        errors = ((recon - X) ** 2).mean(dim=1).cpu().numpy()
    return errors

def score_split(model, df, feature_cols, device, split_name):
    errors = compute_reconstruction_error(model, df, feature_cols, device)
    out = pd.DataFrame({
        "row_id": np.arange(len(df)),
        "Class": df["Class"].values,
        "reconstruction_error": errors,
    })
    print(f"{split_name}: benign mean={out[out.Class==0].reconstruction_error.mean():.4f}, "
          f"anomaly mean={out[out.Class==1].reconstruction_error.mean():.4f} "
          f"(n_benign={sum(out.Class==0)}, n_anomaly={sum(out.Class==1)})")
    return out

print("=== AE — scenario_T reconstruction error by split/class ===")
ae_cic_T_scores = {
    "id":  score_split(ae_cic_T, cic_scenarios["scenario_T"]["id"],  CIC_FEATURES, DEVICE, "ID"),
    "id2": score_split(ae_cic_T, cic_scenarios["scenario_T"]["id2"], CIC_FEATURES, DEVICE, "ID2"),
    "ood": score_split(ae_cic_T, cic_scenarios["scenario_T"]["ood"], CIC_FEATURES, DEVICE, "OOD-temporal"),
}

=== AE — scenario_T reconstruction error by split/class ===
ID: benign mean=0.0079, anomaly mean=1.4269 (n_benign=195004, n_anomaly=101453)
ID2: benign mean=0.0164, anomaly mean=1.4286 (n_benign=195004, n_anomaly=101453)
OOD-temporal: benign mean=0.0313, anomaly mean=1.2459 (n_benign=795033, n_anomaly=222835)


In [11]:
def error_distribution_summary(model, df, feature_cols, device, split_name):
    errors = compute_reconstruction_error(model, df, feature_cols, device)
    benign_errors = errors[df["Class"].values == 0]
    print(f"{split_name} (benign, n={len(benign_errors)}):")
    print(f"  mean={benign_errors.mean():.5f}  median={np.median(benign_errors):.5f}  "
          f"p95={np.percentile(benign_errors,95):.5f}  p99={np.percentile(benign_errors,99):.5f}  "
          f"max={benign_errors.max():.5f}")

for split_name, split_df in [("ID", cic_scenarios["scenario_T"]["id"]),
                              ("ID2", cic_scenarios["scenario_T"]["id2"])]:
    error_distribution_summary(ae_cic_T, split_df, CIC_FEATURES, DEVICE, split_name)

ID (benign, n=195004):
  mean=0.00791  median=0.00061  p95=0.01992  p99=0.09194  max=55.23668
ID2 (benign, n=195004):
  mean=0.01637  median=0.00062  p95=0.01955  p99=0.09019  max=867.38428


In [12]:
AE_T_DIR = os.path.join(OUTPUT_DIR, "ae_cic_scenario_T")
os.makedirs(AE_T_DIR, exist_ok=True)

torch.save(ae_cic_T.state_dict(), os.path.join(AE_T_DIR, "model.pt"))

ae_cic_T_config = {
    "model": "Autoencoder",
    "dataset": "cic_ids2017",
    "scenario": "scenario_T (temporal)",
    "architecture": ARCH_CONFIG_CIC,
    "training": TRAIN_CONFIG_CIC,
    "stopped_epoch": stopped_epoch_ae_cic_T,
    "best_val_loss": best_val_loss_ae_cic_T,
    "fit_size": len(splits_for_training["scenario_T"]["fit"]),
    "val_size": len(splits_for_training["scenario_T"]["val"]),
    "note": "ID-vs-ID2 mean reconstruction error diverges due to extreme tail outliers (max 55 vs 867); medians near-identical (0.00061 vs 0.00062). Use median/trimmed mean for degradation comparisons in NB6/NB7, not raw mean.",
}
with open(os.path.join(AE_T_DIR, "config.json"), "w") as f:
    json.dump(ae_cic_T_config, f, indent=2)

for split_name, scores_df in ae_cic_T_scores.items():
    scores_df.to_parquet(os.path.join(AE_T_DIR, f"scores_{split_name}.parquet"), index=False)

print("Saved to:", AE_T_DIR)
print(os.listdir(AE_T_DIR))

Saved to: /kaggle/working/models/ae_cic_scenario_T
['scores_ood.parquet', 'model.pt', 'scores_id.parquet', 'config.json', 'scores_id2.parquet']


In [13]:
import shutil
shutil.make_archive("/kaggle/working/models_archive_T", "zip", "/kaggle/working/models")
print("Size (MB):", os.path.getsize("/kaggle/working/models_archive_T.zip") / (1024*1024))

Size (MB): 8.046927452087402
